### 데이터 뽑아오기

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================
# 1. 티커 매핑
# =========================================
tickers = {
    'KOSPI 200': '^KS200',          # 기준 자산
    'USD/KRW': 'KRW=X',
    'Gold Spot': 'GC=F',
    'KOSDAQ': '^KQ11',
    'Brent Crude Oil': 'BZ=F',
    'NASDAQ': '^IXIC'               # 추가
}

# =========================================
# 2. 야후 파이낸스 데이터 다운로드
# =========================================
price_data = yf.download(
    list(tickers.values()),
    start='2008-01-01',
    end='2026-03-01',
    auto_adjust=False,
    progress=False
)

# ticker -> 자산명 역매핑
name_mapping = {v: k for k, v in tickers.items()}

# =========================================
# 3. Close 데이터만 먼저 정리
#    (파생변수는 전부 Close 기준으로 만들 예정)
# =========================================
close_data = price_data['Close'].rename(columns=name_mapping)

# KOSPI 200 Close만 남기고,
# 기존의 중복 변수명 'KOSPI 200'은 제거하기 위해 이름 변경
close_data = close_data.rename(columns={'KOSPI 200': 'KOSPI 200 Close'})

# =========================================
# 4. KOSPI 200의 OHLC 및 Volume 추가
# =========================================
kospi_ticker = tickers['KOSPI 200']

ohlc_data = pd.DataFrame(index=price_data.index)
ohlc_data['KOSPI 200 Open'] = price_data['Open'][kospi_ticker]
ohlc_data['KOSPI 200 High'] = price_data['High'][kospi_ticker]
ohlc_data['KOSPI 200 Low'] = price_data['Low'][kospi_ticker]
ohlc_data['KOSPI 200 Volume'] = price_data['Volume'][kospi_ticker]
# Close는 이미 있으므로 여기서 따로 또 만들지 않음

# =========================================
# 5. Close 데이터 + OHLC 결합
# =========================================
data = pd.concat([close_data, ohlc_data], axis=1)

# =========================================
# 6. VKOSPI 및 KODEX 200 불러오기
# =========================================
vkospi = pd.read_csv(r'..\..\data\raw\VKOSPI_2008.csv', encoding='utf-8-sig')

vkospi['Date'] = pd.to_datetime(vkospi['Date'])
vkospi = vkospi[['Date', 'VKOSPI_Close', 'VKOSPI_Change']]
vkospi = vkospi.rename(columns={'VKOSPI_Change': 'VKOSPI_Change(%)'})

vkospi = vkospi.set_index('Date').sort_index()

kodex = pd.read_csv(r'..\..\data\raw\KODEX_2009.csv', encoding='utf-8-sig')
kodex['Date'] = pd.to_datetime(kodex['Date'])
kodex = kodex[['Date', 'KODEX 200_Close', 'KODEX 200_NAV', 'KODEX 200_Premium']]
kodex = kodex.set_index('Date').sort_index()

data.index = pd.to_datetime(data.index)

data = data.join(vkospi, how='left')
data = data.join(kodex, how='left')

# =========================================
# 7. KOSPI 200 기본 파생변수 추가
#    로그 수익률(%) = ln(P_t / P_{t-1}) * 100
# =========================================

data['KOSPI 200 Return'] = np.log(data['KOSPI 200 Close'] / data['KOSPI 200 Close'].shift(1)) * 100
data['KOSPI 200 lagged_1_return(%)'] = data['KOSPI 200 Return'].shift(1)
data['KOSPI 200 lagged_2_return(%)'] = data['KOSPI 200 Return'].shift(2)

# =========================================
# 8. KOSPI 200 영업일 기준으로 정렬
#    기준 인덱스: KOSPI 200 Close가 존재하는 날짜
# =========================================
reference_index = data['KOSPI 200 Close'].dropna().index
data = data.reindex(reference_index).sort_index()

# 앞의 결측은 그대로 두고, 중간 결측은 forward fill
data = data.ffill()

# =========================================
# 9. 2008 전체 데이터 백업
#    (파생변수 계산용 원본)
# =========================================
data_2008 = data.copy()

# =========================================
# 10. 분석 시작일 이후 데이터
#     data_2009는 결측 없이 사용 가능하게 처리
# =========================================
data_2009 = data.loc['2009-04-17':].copy()

#data_2008과 data_2009을 csv파일로 저장하는 코드
data_2008.to_csv(r'..\..\data\raw\data_2008.csv', index=True,encoding="utf-8-sig") # 2008년부터 데이터
data_2009.to_csv(r'..\..\data\raw\data_2009.csv', index=True,encoding="utf-8-sig") # 2009년 4월 17일부터 데이터